In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import (
    chi2_contingency, 
    mannwhitneyu, 
    kruskal, 
    shapiro
)
from statsmodels.stats.proportion import proportions_ztest

# Wnioskowanie statystyczne wymaga oryginalnych etykiet i rozkładów empirycznych
df = pd.read_csv('online_shoppers_intention.csv')

# Ustalenie poziomu istotności statystycznej
ALPHA = 0.05

print("--- WNIOSKOWANIE STATYSTYCZNE ---\n")

# =====================================================================
# TEST 1: Analiza lojalności użytkowników (Chi-kwadrat)
# =====================================================================
print("1. Test niezależności Chi-kwadrat: VisitorType vs Revenue")
contingency_table = pd.crosstab(df['VisitorType'], df['Revenue'])
chi2_stat, p_val_chi2, dof, expected = chi2_contingency(contingency_table)

print(f"Statystyka Chi2: {chi2_stat:.4f}, p-value: {p_val_chi2:.4e}")
if p_val_chi2 < ALPHA:
    print("Wynik: Odrzucenie H0. Istnieje statystycznie istotna zależność między typem odwiedzającego a konwersją.\n")
else:
    print("Wynik: Brak podstaw do odrzucenia H0.\n")


# =====================================================================
# WERYFIKACJA ZAŁOŻEŃ: Test normalności Shapiro-Wilka dla PageValues
# =====================================================================
# Ze względu na ograniczenia obliczeniowe testu Shapiro dla dużych prób (N>5000) losujemy podpróbę.
sample_page_values = df['PageValues'].sample(n=4999, random_state=42)
stat_shapiro, p_val_shapiro = shapiro(sample_page_values)
print("Weryfikacja normalności rozkładu PageValues (Test Shapiro-Wilka):")
print(f"p-value: {p_val_shapiro:.4e} -> Rozkład różny od normalnego, wymóg użycia testów nieparametrycznych.\n")


# =====================================================================
# TEST 2: Ewaluacja istotności metryki Page Value (U Manna-Whitneya)
# =====================================================================
print("2. Test U Manna-Whitneya: PageValues dla Revenue=True vs Revenue=False")
page_values_revenue_true = df[df['Revenue'] == True]['PageValues']
page_values_revenue_false = df[df['Revenue'] == False]['PageValues']

u_stat, p_val_u = mannwhitneyu(page_values_revenue_true, page_values_revenue_false, alternative='two-sided')

print(f"Statystyka U: {u_stat:.4f}, p-value: {p_val_u:.4e}")
if p_val_u < ALPHA:
    print("Wynik: Odrzucenie H0. Wartość podstron (PageValues) pochodzi z różnych rozkładów dla obu grup.")
    print(f"Średnia dla Revenue=True: {page_values_revenue_true.mean():.2f}")
    print(f"Średnia dla Revenue=False: {page_values_revenue_false.mean():.2f}\n")
else:
    print("Wynik: Brak podstaw do odrzucenia H0.\n")


# =====================================================================
# TEST 3: Efekt Weekendu (Test Z dla proporcji)
# =====================================================================
print("3. Test Z dla dwóch proporcji: Konwersja w Weekend vs Dni robocze")
successes = np.array([
    df[(df['Weekend'] == True) & (df['Revenue'] == True)].shape[0],
    df[(df['Weekend'] == False) & (df['Revenue'] == True)].shape[0]
])
nobs = np.array([
    df[df['Weekend'] == True].shape[0],
    df[df['Weekend'] == False].shape[0]
])

z_stat, p_val_z = proportions_ztest(count=successes, nobs=nobs)

print(f"Statystyka Z: {z_stat:.4f}, p-value: {p_val_z:.4e}")
if p_val_z < ALPHA:
    prop_weekend = successes[0] / nobs[0]
    prop_workday = successes[1] / nobs[1]
    print("Wynik: Odrzucenie H0. Proporcje konwersji różnią się statystycznie.")
    print(f"Współczynnik konwersji - Weekend: {prop_weekend:.4f}, Dni robocze: {prop_workday:.4f}\n")
else:
    print("Wynik: Brak podstaw do odrzucenia H0. Weekend nie wpływa istotnie na odsetek konwersji.\n")


# =====================================================================
# TEST 4: Wpływ zdarzeń sezonowych (Kruskal-Wallis)
# =====================================================================
print("4. Test Kruskala-Wallisa: SpecialDay vs ProductRelated_Duration")
# Agregacja wektorów czasu trwania do listy na podstawie unikalnych wartości SpecialDay
groups = [df[df['SpecialDay'] == val]['ProductRelated_Duration'].values for val in df['SpecialDay'].unique()]

h_stat, p_val_h = kruskal(*groups)

print(f"Statystyka H: {h_stat:.4f}, p-value: {p_val_h:.4e}")
if p_val_h < ALPHA:
    print("Wynik: Odrzucenie H0. Rozkład czasu spędzonego na stronach produktowych różni się w zależności od bliskości daty specjalnej.\n")
else:
    print("Wynik: Brak podstaw do odrzucenia H0.\n")

--- WNIOSKOWANIE STATYSTYCZNE ---

1. Test niezależności Chi-kwadrat: VisitorType vs Revenue
Statystyka Chi2: 135.2519, p-value: 4.2699e-30
Wynik: Odrzucenie H0. Istnieje statystycznie istotna zależność między typem odwiedzającego a konwersją.

Weryfikacja normalności rozkładu PageValues (Test Shapiro-Wilka):
p-value: 6.8030e-86 -> Rozkład różny od normalnego, wymóg użycia testów nieparametrycznych.

2. Test U Manna-Whitneya: PageValues dla Revenue=True vs Revenue=False
Statystyka U: 17166757.0000, p-value: 0.0000e+00
Wynik: Odrzucenie H0. Wartość podstron (PageValues) pochodzi z różnych rozkładów dla obu grup.
Średnia dla Revenue=True: 27.26
Średnia dla Revenue=False: 1.98

3. Test Z dla dwóch proporcji: Konwersja w Weekend vs Dni robocze
Statystyka Z: 3.2530, p-value: 1.1420e-03
Wynik: Odrzucenie H0. Proporcje konwersji różnią się statystycznie.
Współczynnik konwersji - Weekend: 0.1740, Dni robocze: 0.1489

4. Test Kruskala-Wallisa: SpecialDay vs ProductRelated_Duration
Statystyka H: